# Swim Club Analytics — SQL Showcase

This notebook runs every analytical query in the project and shows the result next to the SQL. Each section tackles a real business question a swim club coach, parent, or administrator would ask.

**Dataset:** 85 swimmers · 6 divisions · 4 weekly racemeets · 960 race-lane results · 149 nominations.

**Techniques demonstrated:** window functions (LAG, DENSE_RANK, ROW_NUMBER, running aggregates), CTEs, conditional aggregation, anti-joins for gap analysis.

**How this notebook is structured:**
1. Setup — connect to SQLite, define a `show` helper
2. Four business questions the dashboard will answer
3. Advanced technique showcase
4. A multi-stroke participation tracker showcasing CTE composition

---
## Setup

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH = Path('..') / 'swim_club.db'
conn = sqlite3.connect(DB_PATH)
conn.execute('PRAGMA foreign_keys = ON;')

def show(sql, limit=None):
    """Run SQL and return a DataFrame."""
    df = pd.read_sql_query(sql, conn)
    return df.head(limit) if limit else df

# Confirm the 14 tables and 6 views exist
tables = show("SELECT name, type FROM sqlite_master WHERE type IN ('table','view') ORDER BY type, name")
tables

name,type


---
## Business Question 1 — Coach effectiveness

> *"Which coach's division performs best?"*

Raw total points would favour whichever division has the most swimmers, so we normalise by swimmer count. We also include the DNF/DSQ rate as a discipline proxy — a coach who drills race technique well should have fewer disqualifications.

In [2]:
show("""
SELECT
    RANK() OVER (ORDER BY points_per_swimmer DESC) AS rank,
    coach_name,
    division_label,
    swimmer_count,
    total_points,
    points_per_swimmer,
    wins,
    dnf_dsq_rate_pct
FROM v_coach_scorecard
ORDER BY rank
""")

rank,coach_name,division_label,swimmer_count,total_points,points_per_swimmer,wins,dnf_dsq_rate_pct
1,Ram Badhan,14F,13,668,51.38,20,0.78
2,Jhon Green,15F,13,650,50.00,20,1.67
3,Garry Singh,15M,14,671,47.93,20,1.52
4,Rachel Davis,13M,14,663,47.36,20,1.56
5,William Anthony,14M,15,664,44.27,20,0.00
6,Jackson Reacher,13F,16,692,43.25,20,1.43


**Reading the result:** Ram Badhan's 14-female division tops points-per-swimmer despite having fewer swimmers than some others. William Anthony's 14-male division stands out on discipline with a 0% DNF/DSQ rate across 124 races.

---
## Business Question 2 — Division leaderboards

> *"Who's winning each division this season?"*

`DENSE_RANK()` partitioned by division means ties share a rank without skipping the next one. A tie-break on `wins DESC` ensures the ranking reflects outcomes, not just participation.

In [3]:
# Show top 3 of each division
show("""
WITH ranked AS (
    SELECT
        division_label,
        DENSE_RANK() OVER (
            PARTITION BY divisionID
            ORDER BY total_points DESC, wins DESC
        ) AS division_rank,
        swimmer_name,
        coach_name,
        races_completed,
        total_points,
        wins
    FROM v_season_points
    WHERE total_points > 0
)
SELECT * FROM ranked
WHERE division_rank <= 3
ORDER BY division_label, division_rank
""")

division_label,division_rank,swimmer_name,coach_name,races_completed,total_points,wins
13F,1,Maria Leon,Jackson Reacher,12,72,3
13F,2,Jessy Lee,Jackson Reacher,12,66,3
13F,3,Charlotte Amelia,Jackson Reacher,11,65,4
13M,1,Tom Petersen,Rachel Davis,12,71,2
13M,2,Ben Chilwell,Rachel Davis,12,70,2
13M,3,Michael Lee,Rachel Davis,12,60,3
14F,1,Stephanie Atkinson,Ram Badhan,12,72,4
14F,2,Priyanka Chopra,Ram Badhan,12,68,1
14F,3,Margaret Jones,Ram Badhan,12,65,1
14M,1,Arafat Majumder,William Anthony,12,66,4


---
## Business Question 3 — Swimmer performance over time

> *"Is this swimmer improving, plateauing, or slowing down?"*

This is where `LAG()` shines. For each freestyle swim, we compare the time to the same swimmer's previous week. Positive delta = faster this week.

In [4]:
show("""
WITH swimmer_times AS (
    SELECT
        swimmer_name,
        division_label,
        week,
        time_seconds,
        LAG(time_seconds) OVER (
            PARTITION BY swimmerID, strokeType
            ORDER BY week
        ) AS prev_week_time
    FROM v_weekly_trends
    WHERE strokeType = 'freestyle'
)
SELECT
    swimmer_name,
    division_label,
    week,
    time_seconds                          AS time_this_week,
    prev_week_time,
    ROUND(prev_week_time - time_seconds, 2) AS improvement_s
FROM swimmer_times
WHERE prev_week_time IS NOT NULL
ORDER BY improvement_s DESC
LIMIT 10
""")

swimmer_name,division_label,week,time_this_week,prev_week_time,improvement_s
Sarita KC,14F,4,49.24,71.90,22.66
Dalia Frezer,14F,4,50.84,70.09,19.25
Stephanie Atkinson,14F,4,49.08,65.86,16.78
Isla Fisher,14F,2,49.47,66.25,16.78
Jaxson Chan,15M,4,43.52,58.46,14.94
Charlotte Amelia,13F,3,56.48,70.67,14.19
Sarita KC,14F,2,48.91,62.69,13.78
Hira Khan,13F,4,63.19,76.89,13.70
Christie Brown,15F,2,49.01,62.65,13.64
Danny Oscar,15M,2,43.78,57.35,13.57


---
## Business Question 4 — Nomination vs actual participation

> *"Is our nomination process working? Are swimmers racing events they didn't sign up for, or no-showing events they did?"*

This is the data-quality story the NominationInstance entity unlocks.

In [5]:
show("""
SELECT
    SUM(CASE WHEN status = 'Nominated and raced'        THEN 1 ELSE 0 END) AS nominated_and_raced,
    SUM(CASE WHEN status = 'Nominated but did not race' THEN 1 ELSE 0 END) AS nominated_no_show,
    SUM(CASE WHEN status = 'Raced but did not nominate' THEN 1 ELSE 0 END) AS raced_without_nominating,
    ROUND(100.0 * SUM(CASE WHEN status = 'Nominated but did not race' THEN 1 ELSE 0 END)
                / NULLIF(SUM(CASE WHEN status IN ('Nominated and raced','Nominated but did not race')
                                  THEN 1 ELSE 0 END), 0), 2)
        AS no_show_rate_pct
FROM v_nomination_audit
""")

nominated_and_raced,nominated_no_show,raced_without_nominating,no_show_rate_pct
136,13,636,8.72


**Reading the result:** 636 races happened without a matching nomination — more than 4x the number of nominations filed. This is strong evidence that nomination capture is incomplete. A real BA would flag this to stakeholders: either swimmers are skipping the nomination step, or the nomination data isn't being captured end-to-end. Either way it's actionable.

Separately, the no-show rate on *actual* nominations is only 8.72% — suggesting the nomination process, when used, is reasonably reliable.

---
# Advanced technique showcase

A handful of queries that go beyond the business questions to demonstrate specific SQL patterns.

### Running average — consistency vs volatility

Window function with an unbounded preceding frame gives us a true running average. Useful to see whether a swimmer's recent times are above or below their season average.

In [6]:
show("""
SELECT
    swimmer_name,
    week,
    time_seconds,
    ROUND(AVG(time_seconds) OVER (
        PARTITION BY swimmerID
        ORDER BY week
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ), 2) AS running_avg,
    COUNT(*) OVER (
        PARTITION BY swimmerID
        ORDER BY week
    ) AS races_so_far
FROM v_weekly_trends
WHERE strokeType = 'freestyle'
  AND swimmer_name = 'Ross Geller'
ORDER BY week
""")

swimmer_name,week,time_seconds,running_avg,races_so_far
Ross Geller,1,55.81,55.81,1
Ross Geller,2,55.55,55.68,2
Ross Geller,3,66.71,59.36,3
Ross Geller,4,55.91,58.49,4


### Top performer per division per week

Does the same swimmer dominate their division every week, or does it rotate? `DENSE_RANK()` partitioned by both division and week answers cleanly.

In [7]:
show("""
WITH weekly_points AS (
    SELECT
        divisionID, division_label, week,
        swimmer_name,
        SUM(points) AS weekly_points
    FROM v_race_results
    GROUP BY divisionID, division_label, week, swimmerID, swimmer_name
),
ranked AS (
    SELECT *, DENSE_RANK() OVER (PARTITION BY divisionID, week ORDER BY weekly_points DESC) AS rnk
    FROM weekly_points
)
SELECT division_label, week, swimmer_name, weekly_points
FROM ranked WHERE rnk = 1
ORDER BY division_label, week
""")

division_label,week,swimmer_name,weekly_points
13F,1,Monica Bing,17
13F,1,Maria Leon,17
13F,2,Charlotte Amelia,20
13F,3,Charlotte Amelia,19
13F,4,Jessy Lee,21
13M,1,Ben Chilwell,23
13M,2,Tom Petersen,21
13M,3,Michael Lee,20
13M,4,Ben Chilwell,16
13M,4,Atif Anwar,16


### DNF / DSQ anomaly detection

Swimmers whose failure rate exceeds the season average — worth a coach's attention.

In [8]:
show("""
WITH swimmer_rates AS (
    SELECT
        swimmerID, swimmer_name, division_label,
        COUNT(*) AS races,
        SUM(CASE WHEN positionResult IN ('DNF','DSQ') THEN 1 ELSE 0 END) AS dnf_dsq,
        1.0 * SUM(CASE WHEN positionResult IN ('DNF','DSQ') THEN 1 ELSE 0 END) / COUNT(*) AS dnf_rate
    FROM v_race_results
    GROUP BY swimmerID, swimmer_name, division_label
    HAVING COUNT(*) >= 3
),
season_avg AS (
    SELECT 1.0 * SUM(dnf_dsq) / SUM(races) AS avg_rate FROM swimmer_rates
)
SELECT
    swimmer_name, division_label, races, dnf_dsq,
    ROUND(100.0 * dnf_rate, 2) AS swimmer_rate_pct,
    ROUND(100.0 * (SELECT avg_rate FROM season_avg), 2) AS season_avg_pct
FROM swimmer_rates
WHERE dnf_rate > (SELECT avg_rate FROM season_avg)
ORDER BY dnf_rate DESC
""")

swimmer_name,division_label,races,dnf_dsq,swimmer_rate_pct,season_avg_pct
Madison Heart,14F,8,1,12.50,1.17
Farhana Sheikh,13F,8,1,12.50,1.17
Andrew Donald,15M,8,1,12.50,1.17
Durand Jones,13M,8,1,12.50,1.17
Daley Blind,13M,8,1,12.50,1.17
Trevor Huffman,15M,8,1,12.50,1.17
Catherine Warner,13F,8,1,12.50,1.17
Olovia Robinson,15F,12,1,8.33,1.17
Alisha Ben,15F,12,1,8.33,1.17


### Lane utilisation per event

Each race has 8 available lanes. How well does each event fill them?

In [9]:
show("""
SELECT
    et.eventDescription,
    et.strokeType,
    COUNT(*) AS total_lane_slots,
    SUM(CASE WHEN rli.swimmerID IS NOT NULL THEN 1 ELSE 0 END) AS lanes_used,
    ROUND(100.0 * SUM(CASE WHEN rli.swimmerID IS NOT NULL THEN 1 ELSE 0 END)
                / COUNT(*), 1) AS utilisation_pct
FROM RaceLaneInstance rli
JOIN RaceInstance ri ON rli.raceInstanceID = ri.raceInstanceID
JOIN RacemeetEventInstance rmei ON ri.racemeetEventInstanceID = rmei.racemeetEventInstanceID
JOIN EventType et ON rmei.eventTypeID = et.eventTypeID
GROUP BY et.eventDescription, et.strokeType
ORDER BY utilisation_pct DESC
""")

eventDescription,strokeType,total_lane_slots,lanes_used,utilisation_pct
50m breaststroke,breaststroke,192,160,83.3
50m butterfly,butterfly,192,160,83.3
50m backstroke,backstroke,192,156,81.3
4 x 50m individual medley,medley,192,148,77.1
50m freestyle,freestyle,192,148,77.1


---
## Multi-stroke participation tracker

An operational question a club admin might ask:

> *During Racemeet 2, find the 13-year-old boy who swam freestyle, butterfly, AND medley, and list his results.*

Solved cleanly using CTEs and the `v_race_results` view — the `HAVING COUNT(DISTINCT...)` pattern is a useful template for any cross-category audit query.

In [10]:
show("""
WITH target_swimmers AS (
    SELECT swimmerID
    FROM v_race_results
    WHERE week = 2
      AND swimmer_age = 13
      AND swimmer_gender = 'M'
      AND strokeType IN ('freestyle','butterfly','medley')
    GROUP BY swimmerID
    HAVING COUNT(DISTINCT strokeType) = 3
    LIMIT 1
)
SELECT
    rr.swimmer_name AS competitor,
    rr.eventDescription,
    rr.race_start_time,
    rr.positionResult AS result,
    rr.time_seconds,
    rr.points
FROM v_race_results rr
JOIN target_swimmers ts ON rr.swimmerID = ts.swimmerID
WHERE rr.week = 2
ORDER BY rr.race_start_time
""")

competitor,eventDescription,race_start_time,result,time_seconds,points
Atif Anwar,50m freestyle,5:30:00 PM,3rd,66.33,6
Atif Anwar,50m butterfly,6:45:00 PM,6th,88.44,3
Atif Anwar,4 x 50m individual medley,8:00:00 PM,4th,321.81,5


The view layer plus CTEs make this kind of multi-criteria filter expressive without sacrificing readability.

---
## What Power BI sees

Power BI connects directly to this SQLite database via the ODBC driver (see `docs/powerbi-setup.md`) and loads the six views — not the raw tables. The dashboard is four pages:

| Page | Primary view | Business question |
|---|---|---|
| Season Overview | `v_coach_scorecard`, `v_season_points` | Who won, how did each division perform? |
| Swimmer Trending | `v_weekly_trends` | Which swimmers improved, plateaued, regressed? |
| Coach Effectiveness | `v_coach_scorecard` | Which coach's methods translate to results? |
| Data Quality | `v_nomination_audit` | Are our operational processes being followed? |

In [11]:
conn.close()